# Interactive plotting of embedding transforms

Loads pre-computed t-SNE / UMAP coordinates from the `gelos.analysis` cache and renders them interactively with Plotly so you can zoom, pan, box-/lasso-select, and hover for chip metadata.

Example uses `configs/exp001_prithvi300.yaml`. Run `gelos analysis -y <config>` first to populate the cache, then swap `yaml_path` to plot any other experiment.

NOTE: This is llm-generated as a sanity check for the umap transforms for this project

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yaml
from dotenv import load_dotenv
from plotly.subplots import make_subplots

from gelos.analysis import build_prefix, load_chip_tracker

load_dotenv("../.env")


## Configure run

Data paths come from `RAW_PATH` / `PROCESSED_PATH` env vars (set in-container by `compose.yml`; loaded on host from `../.env` by `load_dotenv`). Transform CSVs are read from the `gelos.analysis` cache under `PROCESSED_PATH`. Config paths are resolved relative to the notebook.


In [ ]:
raw_data_dir = Path(os.environ["RAW_PATH"])
processed_dir = Path(os.environ["PROCESSED_PATH"])
yaml_path = Path("../configs/exp001_prithvi300.yaml")

with open(yaml_path) as f:
    yaml_config = yaml.safe_load(f)

data_version = yaml_config["data_version"]
chip_tracker_file = yaml_config["chip_tracker"]
chip_id_column = yaml_config["chip_id_column"]
category_column = yaml_config["style"]["category_column"]
experiment_name = yaml_config["experiment_name"]

data_root = raw_data_dir / data_version
config_stem = yaml_path.stem
output_dir = processed_dir / data_version / config_stem

chip_gdf = load_chip_tracker(data_root / chip_tracker_file).set_index(chip_id_column)
print(f"loaded {len(chip_gdf)} chips for {experiment_name} ({data_version})")


## Select embedding layer & extraction strategy

Edit `embedding_layer` / `strategy_key` to plot a different slice.

In [ ]:
print("Available embedding layer directories:")
for d in sorted(output_dir.glob("*")):
    print(f"  {d.name}")

print("\nAvailable extraction strategies in config:")
for k, v in yaml_config["embedding_extraction_strategies"].items():
    print(f"  {k}  ({v.get('title', k)})")

In [ ]:
embedding_layer = "layer_23"
strategy_key = "all_steps_of_middle_patch"

strategy_cfg = yaml_config["embedding_extraction_strategies"][strategy_key]
strategy_title = strategy_cfg.get("title", strategy_key)
layer_dir = output_dir / embedding_layer
prefix = build_prefix(config_stem, strategy_key, embedding_layer)

print(f"strategy: {strategy_title}")
print(f"layer:    {embedding_layer}")
print(f"reading:  {layer_dir}")
print(f"prefix:   {prefix}")

## Load cached transform results

Reads the `{prefix}_tsne.csv` / `{prefix}_umap.csv` files produced by `gelos.analysis` (columns: `id`, `dim_0`, `dim_1`). If the files are missing, run `gelos analysis -y ../configs/{config_stem}.yaml` first to populate the cache.

In [ ]:
def _load_transform_csv(layer_dir: Path, prefix: str, transform_type: str) -> tuple[np.ndarray, list[int]]:
    """Load a cached transform CSV (columns: id, dim_0, dim_1, ...)."""
    path = layer_dir / f"{prefix}_{transform_type}.csv"
    df = pd.read_csv(path)
    chip_indices = df["id"].tolist()
    coords = df[[c for c in df.columns if c != "id"]].to_numpy()
    return coords, chip_indices


tsne_result, chip_indices = _load_transform_csv(layer_dir, prefix, "tsne")
print(f"tsne: {tsne_result.shape}  |  unique chips: {len(set(chip_indices))}")

In [ ]:
umap_result, umap_chip_indices = _load_transform_csv(layer_dir, prefix, "umap")
assert umap_chip_indices == chip_indices, "tsne and umap chip indices must align"
print(f"umap: {umap_result.shape}")

## Build a tidy DataFrame for plotting

Joins transform coordinates against the chip tracker so hover tooltips can show land-cover class.

In [ ]:
style_cfg = yaml_config["style"]
color_map_raw = style_cfg["colors"]
label_map_raw = style_cfg["labels"]

# yaml keys come in as strings (e.g. "1"); coerce to the dtype actually stored in chip_gdf[category_column]
sample_cat_val = chip_gdf[category_column].iloc[0]
cast = type(sample_cat_val.item()) if isinstance(sample_cat_val, np.generic) else type(sample_cat_val)
color_map = {cast(k): v for k, v in color_map_raw.items()}
label_map = {cast(k): v for k, v in label_map_raw.items()}

categories = chip_gdf[category_column].loc[chip_indices].to_numpy()
labels = pd.Series(categories).map(label_map).fillna(pd.Series(categories).astype(str)).to_numpy()

plot_df = pd.DataFrame({
    "chip_id": chip_indices,
    "category": categories,
    "label": labels,
    "tsne_x": tsne_result[:, 0],
    "tsne_y": tsne_result[:, 1],
    "umap_x": umap_result[:, 0],
    "umap_y": umap_result[:, 1],
})

# label-keyed colour map for plotly (it groups by the column passed as `color`)
label_color_map = {label_map[k]: v for k, v in color_map.items() if k in label_map}
plot_df.head()

## Interactive t-SNE

Use the mode bar (top-right of the figure) to **pan**, **zoom**, **box-select**, **lasso-select**, or **reset axes**. Click legend entries to hide/show classes. Hover for chip metadata.

In [ ]:
fig_tsne = px.scatter(
    plot_df,
    x="tsne_x",
    y="tsne_y",
    color="label",
    color_discrete_map=label_color_map,
    hover_data={"chip_id": True, "label": True, "tsne_x": ":.2f", "tsne_y": ":.2f"},
    title=f"t-SNE — {experiment_name} · {strategy_title} · {embedding_layer}",
    height=650,
)
fig_tsne.update_traces(marker=dict(size=6, opacity=0.75, line=dict(width=0)))
fig_tsne.update_layout(
    legend_title_text="Land cover",
    dragmode="pan",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2",
)
fig_tsne.show(config={"scrollZoom": True})

## Interactive UMAP

In [ ]:
fig_umap = px.scatter(
    plot_df,
    x="umap_x",
    y="umap_y",
    color="label",
    color_discrete_map=label_color_map,
    hover_data={"chip_id": True, "label": True, "umap_x": ":.2f", "umap_y": ":.2f"},
    title=f"UMAP — {experiment_name} · {strategy_title} · {embedding_layer}",
    height=650,
)
fig_umap.update_traces(marker=dict(size=6, opacity=0.75, line=dict(width=0)))
fig_umap.update_layout(
    legend_title_text="Land cover",
    dragmode="pan",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
)
fig_umap.show(config={"scrollZoom": True})

## Side-by-side comparison

Same data, two transforms. Legend is shared via `legendgroup` — toggling a class hides it in both panels.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("t-SNE", "UMAP"), horizontal_spacing=0.08)

for label, color in label_color_map.items():
    sub = plot_df[plot_df["label"] == label]
    if sub.empty:
        continue
    customdata = np.stack([sub["chip_id"].to_numpy(), sub["label"].to_numpy()], axis=-1)
    hover = "chip_id=%{customdata[0]}<br>label=%{customdata[1]}<extra></extra>"
    fig.add_trace(
        go.Scattergl(
            x=sub["tsne_x"], y=sub["tsne_y"], mode="markers",
            marker=dict(size=6, color=color, opacity=0.75),
            name=label, legendgroup=label,
            customdata=customdata, hovertemplate=hover,
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scattergl(
            x=sub["umap_x"], y=sub["umap_y"], mode="markers",
            marker=dict(size=6, color=color, opacity=0.75),
            name=label, legendgroup=label, showlegend=False,
            customdata=customdata, hovertemplate=hover,
        ),
        row=1, col=2,
    )

fig.update_layout(
    title=f"{experiment_name} · {strategy_title} · {embedding_layer}",
    height=650,
    legend_title_text="Land cover",
    dragmode="pan",
)
fig.update_xaxes(title_text="t-SNE 1", row=1, col=1)
fig.update_yaxes(title_text="t-SNE 2", row=1, col=1)
fig.update_xaxes(title_text="UMAP 1", row=1, col=2)
fig.update_yaxes(title_text="UMAP 2", row=1, col=2)
fig.show(config={"scrollZoom": True})

## Inspect a region

Pick a coordinate window (read it off the zoomed plot above) and look at the matching chips. For programmatic capture of box-/lasso-selections, swap the figures above for `go.FigureWidget` and attach an `on_selection` callback.

In [ ]:
x_min, x_max = -np.inf, np.inf
y_min, y_max = -np.inf, np.inf

region = plot_df[
    plot_df["umap_x"].between(x_min, x_max) & plot_df["umap_y"].between(y_min, y_max)
]
print(f"{len(region)} points in region")
region["label"].value_counts()